# Task 1: Big Data Processing with PySpark

This notebook demonstrates scalable processing on the supermarket dataset using PySpark.

In [ ]:
import os
import pandas as pd

os.environ['JAVA_HOME'] = 'C:/Program Files/Java/jdk-17'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'python'
os.environ['PYSPARK_PYTHON'] = 'python'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Supermarket Big Data Analysis') \
    .config('spark.driver.host', '127.0.0.1') \
    .getOrCreate()

data_path = 'c:/Users/Admin/OneDrive/Documents/BigData project/SuperMarket Analysis.csv'
pandas_df = pd.read_csv(data_path)
spark_df = spark.read.csv(data_path, header=True, inferSchema=True)

print('Pandas shape:', pandas_df.shape)
print('Spark rows:', spark_df.count())
print('Spark columns:', spark_df.columns)

## Scaling the dataset

To demonstrate scalability, the notebook expands the dataset to 10,000 rows with a Spark `crossJoin`.

In [ ]:
expanded_df = spark_df.repartition(4).cache().crossJoin(spark.range(10))
expanded_count = expanded_df.count()
expanded_df = expanded_df.withColumn('sale_date', F.to_date(F.col('Date'), 'M/d/yyyy')) \
    .withColumn('month', F.date_format('sale_date', 'yyyy-MM'))

branch_sales = expanded_df.groupBy('Branch').agg(
    F.sum('Sales').alias('TotalSales'),
    F.count('*').alias('Transactions')
).orderBy(F.desc('TotalSales'))

product_sales = expanded_df.groupBy('Product line').agg(
    F.sum('Sales').alias('TotalSales')
).orderBy(F.desc('TotalSales'))

payment_mix = expanded_df.groupBy('Payment').agg(
    F.count('*').alias('Transactions')
).orderBy(F.desc('Transactions'))

monthly_sales = expanded_df.groupBy('month').agg(
    F.sum('Sales').alias('TotalSales')
).orderBy('month')

print('Expanded rows:', expanded_count)
print('Top branches:')
branch_sales.show(5, truncate=False)
print('Top product lines:')
product_sales.show(5, truncate=False)
print('Payment mix:')
payment_mix.show(truncate=False)
print('Monthly sales:')
monthly_sales.show(truncate=False)

## Key insights

- Branch-level revenue and transaction counts show where demand is strongest.
- Product line performance can be ranked from the aggregated sales.
- Payment preference and monthly trends are available for operational decisions.